# Turn `bikol_chat_lora.zip` into the Ollama model (about 15 minutes)

Use this when training finished but the free GPU time ran out before the model could be exported.

1. **Runtime → Change runtime type → T4 GPU** (Colab) or **Accelerator → GPU** (Kaggle).
2. **Run all.** When asked, upload `bikol_chat_lora.zip` (on Kaggle: *Add Input* → upload it as a dataset first).
3. Download `lmlinga-bikol-chat-v6.zip`, unzip it, then on your computer:
   `ollama create lmlinga-bikol-chat-v6 -f Modelfile`

In [ ]:
!pip install -q unsloth
import importlib.util
if importlib.util.find_spec("unsloth") is None:
    raise SystemExit("unsloth did NOT install - read the red lines above and send them")
print("unsloth installed OK")

In [ ]:
import os, glob, shutil, zipfile
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
MODEL_NAME = "lmlinga-bikol-chat-v6"
QUANT = "q4_k_m"
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
files = None
if ON_KAGGLE:
    os.makedirs("/tmp/bikol", exist_ok=True)
    found = glob.glob("/kaggle/input/**/bikol_chat_lora*.zip", recursive=True)
    assert found, "Add bikol_chat_lora.zip with Add Input first"
    shutil.copy(found[0], "/tmp/bikol/bikol_chat_lora.zip")
    os.chdir("/tmp/bikol")
else:
    try:
        from google.colab import files
        print("Choose bikol_chat_lora.zip")
        uploaded = files.upload()
        name = next(iter(uploaded))
        if name != "bikol_chat_lora.zip":
            os.replace(name, "bikol_chat_lora.zip")
    except ImportError:
        pass

def give(name):
    if files:
        files.download(name)
    elif ON_KAGGLE:
        shutil.copy(name, "/kaggle/working/")
        print("Saved to Output:", name)

with zipfile.ZipFile("bikol_chat_lora.zip") as z:
    z.extractall("bikol_chat_lora")
print("adapter files:", sorted(os.listdir("bikol_chat_lora"))[:6])

In [ ]:
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name="bikol_chat_lora",   # the saved adapter: the base model is fetched from its config
    max_seq_length=1024,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
print("adapter loaded")

## Step 9 — Save for Ollama and download

Makes a **GGUF** file (Ollama's model format) and a **Modelfile** (Ollama's settings file), then zips them.

**To use it** (after unzipping `lmlinga-bikol-chat.zip`):

```
cd lmlinga-bikol-chat
ollama create lmlinga-bikol-chat -f Modelfile
ollama run lmlinga-bikol-chat "Kumusta ika?"
```

In [ ]:
import glob, shutil

# Small backup of what was learned (LoRA adapter)
model.save_pretrained("bikol_chat_lora")
tokenizer.save_pretrained("bikol_chat_lora")
shutil.make_archive("bikol_chat_lora", "zip", "bikol_chat_lora")

# Full model as GGUF for Ollama (the Unsloth function arguments changed between versions)
OUT = MODEL_NAME
try:
    model.save_pretrained_gguf(OUT, tokenizer, quantization_method=QUANT)
except TypeError:
    model.save_pretrained_gguf(OUT, quantization_type=QUANT.upper())

ggufs = sorted(glob.glob(f"{OUT}*/**/*.gguf", recursive=True) + glob.glob(f"{OUT}*.gguf") + glob.glob("*.gguf"))
ggufs = [g for g in ggufs if "mmproj" not in g.lower()]   # skip the image part, text only
print("GGUF files found:", ggufs)
assert ggufs, "No .gguf file was made - see the error above"
os.makedirs("ollama_export", exist_ok=True)
gguf_name = f"{MODEL_NAME}.{QUANT.upper()}.gguf"
shutil.copy(ggufs[0], os.path.join("ollama_export", gguf_name))

In [ ]:
Q = chr(34) * 3   # three double quotes, used by the Modelfile
MODELFILE = f"""FROM ./{gguf_name}

TEMPLATE {Q}{{{{- range $i, $_ := .Messages }}}}
{{{{- $last := eq (len (slice $.Messages $i)) 1 }}}}
{{{{- if eq .Role "user" }}}}<start_of_turn>user
{{{{ .Content }}}}<end_of_turn>
{{{{ if $last }}}}<start_of_turn>model
{{{{ end }}}}
{{{{- else if eq .Role "assistant" }}}}<start_of_turn>model
{{{{ .Content }}}}{{{{ if not $last }}}}<end_of_turn>
{{{{ end }}}}
{{{{- end }}}}
{{{{- end }}}}{Q}

PARAMETER stop "<end_of_turn>"
PARAMETER temperature 0.3
PARAMETER num_ctx 2048
"""
with open("ollama_export/Modelfile", "w", encoding="utf-8") as f:
    f.write(MODELFILE)
print(MODELFILE)

shutil.make_archive(MODEL_NAME, "zip", "ollama_export")
print("Made:", f"{MODEL_NAME}.zip", "bikol_chat_lora.zip")
for name in [f"{MODEL_NAME}.zip", "bikol_chat_lora.zip"]:
    give(name)